In [12]:
import pandas as pd
import os

PATH = "posts.json"
ORIGIN_PATH = "hf://datasets/ScoutieAutoML/russian-news-telegram-dataset/scoutieRussianNewsTelegramDataset.json"

if not os.path.exists(PATH):
    df.to_json(PATH)
    df.head()

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import MinMaxScaler
from umap import UMAP


MIN_CLUSTER_SIZE = 5
TIME_WEIGHT = 0.3


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_json(path, orient="columns")

    df = df[df["spam"] == "NOT SPAM"]
    df = df[df["language"] == "rus"]
    df = df.dropna(subset=["vector", "create_time", "text"])
    df = df.sort_values("create_time").reset_index(drop=True)

    df["datetime"] = pd.to_datetime(df["create_time"], unit="ms")

    return df


def parse_vectors(df: pd.DataFrame) -> np.ndarray:
    vectors = np.vstack(df["vector"].apply(lambda v: np.array(v, dtype=np.float32)).values)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    return vectors / norms


def normalize_timestamps(df: pd.DataFrame) -> np.ndarray:
    scaler = MinMaxScaler()
    ts = df["create_time"].values.astype(np.float64).reshape(-1, 1)
    return scaler.fit_transform(ts).flatten()


def reduce_dimensions(
    vectors: np.ndarray,
    n_components: int = 50,
    n_neighbors: int = 15,
    random_state: int = 42,
) -> np.ndarray:
    reducer = UMAP(
        n_components=20,
        n_neighbors=15,
        metric="cosine",
        random_state=42,
        low_memory=True,
        verbose=True,
    )
    return reducer.fit_transform(vectors)


def build_feature_matrix(
    vectors_reduced: np.ndarray,
    timestamps_norm: np.ndarray,
    time_weight: float = 0.5,
) -> np.ndarray:
    time_col = (timestamps_norm * time_weight).reshape(-1, 1)
    return np.hstack([vectors_reduced, time_col])


def cluster_events(
    features: np.ndarray,
    min_cluster_size: int = MIN_CLUSTER_SIZE,
    min_samples: int = 5,
) -> np.ndarray:
    clusterer = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
    )
    return clusterer.fit_predict(features)


def extract_ner_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def get_by_label(ners, labels):
        if not isinstance(ners, list):
            return []
        return list({e["lemma"] for e in ners if e.get("label") in labels})

    df["ner_persons"] = df["ners"].apply(lambda x: get_by_label(x, {"PER"}))
    df["ner_orgs"] = df["ners"].apply(lambda x: get_by_label(x, {"ORG"}))
    df["ner_locations"] = df["ners"].apply(lambda x: get_by_label(x, {"LOC", "GPE"}))

    return df


def summarize_clusters(df: pd.DataFrame) -> pd.DataFrame:
    def top_entities(series, n=5):
        all_ents = [e for lst in series for e in lst]
        return [e for e, _ in Counter(all_ents).most_common(n)]

    grouped = df[df["event_cluster"] != -1].groupby("event_cluster")

    summary = grouped.agg(
        post_count=("id", "count"),
        date_start=("datetime", "min"),
        date_end=("datetime", "max"),
        avg_views=("views", "mean"),
        emo_positive=("emo", lambda x: (x == "POSITIVE").sum()),
        emo_negative=("emo", lambda x: (x == "NEGATIVE").sum()),
        emo_neutral=("emo", lambda x: (x == "NEUTRAL").sum()),
    ).reset_index()

    summary["duration_days"] = (summary["date_end"] - summary["date_start"]).dt.days

    ner_agg = grouped.apply(
        lambda g: pd.Series({
            "top_persons": top_entities(g["ner_persons"]),
            "top_orgs": top_entities(g["ner_orgs"]),
            "top_locations": top_entities(g["ner_locations"]),
        })
    ).reset_index()

    summary = summary.merge(ner_agg, on="event_cluster")
    return summary.sort_values("post_count", ascending=False).reset_index(drop=True)


def run_pipeline(
    jsonl_path: str,
    time_weight: float = TIME_WEIGHT,
    min_cluster_size: int = 15,
    umap_components: int = 20,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("Loading data...")
    df = load_data(jsonl_path)
    print(f"Posts after filtering: {len(df)}")

    print("Parsing vectors...")
    vectors = parse_vectors(df)

    print("Reducing dimensions with UMAP (768 -> 50)...")
    vectors_reduced = reduce_dimensions(vectors, n_components=umap_components)
    print(f"Reduced shape: {vectors_reduced.shape}")

    print("Building feature matrix with time...")
    timestamps_norm = normalize_timestamps(df)
    features = build_feature_matrix(vectors_reduced, timestamps_norm, time_weight)

    print("Clustering with HDBSCAN...")
    labels = cluster_events(features, min_cluster_size=min_cluster_size)
    df["event_cluster"] = labels

    n_events = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"Events found:  {n_events}")
    print(f"Noise posts:   {n_noise} ({n_noise / len(df) * 100:.1f}%)")

    print("Extracting NER...")
    df = extract_ner_columns(df)

    print("Building summary...")
    summary = summarize_clusters(df)

    return df, summary


if __name__ == "__main__":
    PATH = "posts.json"

    posts, events = run_pipeline(
        jsonl_path=PATH,
        time_weight=TIME_WEIGHT,
        min_cluster_size=MIN_CLUSTER_SIZE,
        umap_components=20,
    )

    posts.to_json("posts_clustered.json", orient="records", lines=True, force_ascii=False)
    events.to_json("events_summary.json", orient="records", lines=True, force_ascii=False)

    print("\nTop 10 events by post count:")
    cols = ["event_cluster", "post_count", "date_start", "date_end", "duration_days", "top_persons", "top_orgs"]
    print(events[cols].head(10).to_string())

Loading data...
Posts after filtering: 92939
Parsing vectors...
Reducing dimensions with UMAP (768 -> 50)...
UMAP(angular_rp_forest=True, metric='cosine', n_components=20, n_jobs=1, random_state=42, verbose=True)
Thu Apr 30 09:08:21 2026 Construct fuzzy simplicial set
Thu Apr 30 09:08:21 2026 Finding Nearest Neighbors
Thu Apr 30 09:08:21 2026 Building RP forest with 20 trees
Thu Apr 30 09:08:25 2026 NN descent for 17 iterations
	 1  /  17
	 2  /  17
	 3  /  17
	 4  /  17
	 5  /  17
	 6  /  17
	 7  /  17
	Stopping threshold met -- exiting after 7 iterations
Thu Apr 30 09:08:39 2026 Finished Nearest Neighbor Search
Thu Apr 30 09:08:40 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Thu Apr 30 09:09:38 2026 Finished embedding
Reduced shape: (92939, 20)
Building feature matrix with time...
Clustering with HDBSCAN...
Events found:  771
Noise posts:   49970 (53.8%)
Extracting NER...
Building summary...

Top 10 events by post count:
   event_cluster  post_count          date_start            date_end  duration_days                                                                        top_persons                                                  top_orgs
0            659        1115 2022-05-11 17:22:56 2024-10-31 14:13:07            903                 [гладков, вячеслав гладков, белгород, шебекино, александр богомаз]                               [всу, пво, shot, mash, 

In [14]:
import json
import numpy as np
import pandas as pd


def load_clustered(path: str) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    df["datetime"] = pd.to_datetime(df["datetime"], unit="ms")
    df["date"] = df["datetime"].dt.date
    return df


def build_daily_intensity(df: pd.DataFrame) -> pd.DataFrame:
    total_per_day = df.groupby("date")["id"].count().rename("total_posts")

    events = df[df["event_cluster"] != -1].copy()

    daily = (
        events.groupby(["event_cluster", "date"])
        .agg(
            post_count=("id", "count"),
            avg_views=("views", "mean"),
            emo_positive=("emo", lambda x: (x == "POSITIVE").sum()),
            emo_negative=("emo", lambda x: (x == "NEGATIVE").sum()),
            emo_neutral=("emo", lambda x: (x == "NEUTRAL").sum()),
        )
        .reset_index()
    )

    daily = daily.merge(total_per_day, on="date")
    daily["relative_intensity"] = daily["post_count"] / daily["total_posts"]
    daily["date"] = pd.to_datetime(daily["date"])

    return daily


def compute_event_features(daily: pd.DataFrame) -> pd.DataFrame:
    features = []

    for cluster_id, group in daily.groupby("event_cluster"):
        group = group.sort_values("date")

        peak_idx = group["post_count"].idxmax()
        peak_date = group.loc[peak_idx, "date"]
        peak_value = group.loc[peak_idx, "post_count"]

        before_peak = group[group["date"] <= peak_date]
        after_peak = group[group["date"] >= peak_date]

        rise_days = (peak_date - before_peak["date"].min()).days
        decay_days = (after_peak["date"].max() - peak_date).days

        gaps = group["date"].diff().dt.days.dropna()
        max_gap = gaps.max()
        n_bursts = (gaps > 7).sum() + 1

        features.append({
            "event_cluster": cluster_id,
            "total_days": (group["date"].max() - group["date"].min()).days,
            "active_days": len(group),
            "peak_date": peak_date,
            "peak_posts": peak_value,
            "peak_relative": group.loc[peak_idx, "relative_intensity"],
            "rise_days": rise_days,
            "decay_days": decay_days,
            "max_gap_days": max_gap,
            "n_bursts": n_bursts,
            "total_posts": group["post_count"].sum(),
            "total_views": (group["avg_views"] * group["post_count"]).sum(),
        })

    return pd.DataFrame(features).sort_values("total_posts", ascending=False).reset_index(drop=True)


def classify_pattern(row) -> str:
    if row["n_bursts"] >= 3:
        return "multi_burst"
    if row["n_bursts"] == 2:
        return "two_burst"
    if row["decay_days"] > row["rise_days"] * 3:
        return "long_tail"
    if row["rise_days"] > row["decay_days"] * 3:
        return "slow_build"
    return "single_peak"


def run_dynamics(clustered_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("Loading clustered data...")
    df = load_clustered(clustered_path)
    print(f"Total posts: {len(df)}, events: {df['event_cluster'].nunique() - 1}")

    print("Building daily intensity...")
    daily = build_daily_intensity(df)

    print("Computing event features...")
    features = compute_event_features(daily)
    features["pattern"] = features.apply(classify_pattern, axis=1)

    print("\nPattern distribution:")
    print(features["pattern"].value_counts().to_string())

    print("\nTop 10 events:")
    cols = ["event_cluster", "total_posts", "active_days", "peak_date",
            "peak_posts", "n_bursts", "pattern"]
    print(features[cols].head(10).to_string())

    return daily, features


if __name__ == "__main__":
    daily, features = run_dynamics("posts_clustered.json")

    daily.to_json("events_daily.json", orient="records", lines=True, force_ascii=False, date_format="iso")
    features.to_json("events_features.json", orient="records", lines=True, force_ascii=False, date_format="iso")

    print("\nSaved: events_daily.json, events_features.json")

Loading clustered data...
Total posts: 92939, events: 771
Building daily intensity...
Computing event features...

Pattern distribution:
pattern
multi_burst    722
two_burst       29
long_tail       12
single_peak      6
slow_build       2

Top 10 events:
   event_cluster  total_posts  active_days  peak_date  peak_posts  n_bursts      pattern
0            659         1115          277 2024-05-12         162        19  multi_burst
1             80         1087           90 2024-03-23         262        25  multi_burst
2            238          935          434 2024-10-01          28        35  multi_burst
3            629          869          412 2024-10-14          12        17  multi_burst
4            684          757          385 2024-09-29          33        46  multi_burst
5            596          568          291 2023-03-01           8        23  multi_burst
6            700          547          258 2024-08-14          13        36  multi_burst
7            138          408   

In [15]:
import warnings
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")


def load_daily(path: str) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    df["date"] = pd.to_datetime(df["date"])
    return df


def prepare_prophet_df(group: pd.DataFrame) -> pd.DataFrame:
    full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    ts = group.set_index("date")["relative_intensity"].reindex(full_range, fill_value=0)
    return ts.reset_index().rename(columns={"index": "ds", "relative_intensity": "y"})


def train_test_split_ts(df: pd.DataFrame, train_ratio: float = 0.8) -> tuple[pd.DataFrame, pd.DataFrame]:
    split_idx = int(len(df) * train_ratio)
    return df.iloc[:split_idx], df.iloc[split_idx:]


def fit_prophet(train: pd.DataFrame) -> Prophet:
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.1,
        seasonality_prior_scale=1.0,
    )
    model.fit(train)
    return model


def forecast_event(model: Prophet, test: pd.DataFrame) -> pd.DataFrame:
    future = test[["ds"]].copy()
    forecast = model.predict(future)
    return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]


def compute_metrics(actual: pd.Series, predicted: pd.Series) -> dict:
    mae = mean_absolute_error(actual, predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5
    actual_binary = (actual > 0).astype(int)
    pred_binary = (predicted > 0).astype(int)
    tp = ((actual_binary == 1) & (pred_binary == 1)).sum()
    fp = ((actual_binary == 0) & (pred_binary == 1)).sum()
    fn = ((actual_binary == 1) & (pred_binary == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"mae": mae, "rmse": rmse, "precision": precision, "recall": recall, "f1": f1}


def run_forecast(
    daily_path: str,
    min_active_days: int = 30,
    train_ratio: float = 0.8,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("Loading daily data...")
    daily = load_daily(daily_path)

    results = []
    forecasts = []

    clusters = daily["event_cluster"].unique()
    eligible = [
        c for c in clusters
        if len(daily[daily["event_cluster"] == c]) >= min_active_days
    ]
    print(f"Events eligible for forecasting (>={min_active_days} active days): {len(eligible)}")

    for i, cluster_id in enumerate(eligible):
        group = daily[daily["event_cluster"] == cluster_id].sort_values("date")
        prophet_df = prepare_prophet_df(group)

        if len(prophet_df) < min_active_days:
            continue

        train, test = train_test_split_ts(prophet_df, train_ratio)

        if len(test) < 5:
            continue

        try:
            model = fit_prophet(train)
            forecast = forecast_event(model, test)

            actual = test["y"].values
            predicted = np.clip(forecast["yhat"].values, 0, None)

            metrics = compute_metrics(actual, predicted)
            metrics["event_cluster"] = cluster_id
            metrics["train_days"] = len(train)
            metrics["test_days"] = len(test)
            results.append(metrics)

            fc_df = forecast.copy()
            fc_df["actual"] = actual
            fc_df["event_cluster"] = cluster_id
            forecasts.append(fc_df)

            if (i + 1) % 10 == 0:
                print(f"  Processed {i + 1}/{len(eligible)} events...")

        except Exception as e:
            print(f"  Skipped cluster {cluster_id}: {e}")
            continue

    results_df = pd.DataFrame(results)
    forecasts_df = pd.concat(forecasts, ignore_index=True)

    print(f"\nForecasted {len(results_df)} events successfully")
    print("\nAggregate metrics:")
    print(f"  MAE:       {results_df['mae'].mean():.6f}")
    print(f"  RMSE:      {results_df['rmse'].mean():.6f}")
    print(f"  Precision: {results_df['precision'].mean():.3f}")
    print(f"  Recall:    {results_df['recall'].mean():.3f}")
    print(f"  F1:        {results_df['f1'].mean():.3f}")

    print("\nTop 10 best predicted events (by F1):")
    cols = ["event_cluster", "train_days", "test_days", "mae", "rmse", "f1"]
    print(results_df.sort_values("f1", ascending=False)[cols].head(10).to_string())

    return forecasts_df, results_df


if __name__ == "__main__":
    forecasts, metrics = run_forecast(
        daily_path="events_daily.json",
        min_active_days=30,
        train_ratio=0.8,
    )

    forecasts.to_json("forecasts.json", orient="records", lines=True, force_ascii=False, date_format="iso")
    metrics.to_json("forecast_metrics.json", orient="records", lines=True, force_ascii=False, date_format="iso")

    print("\nSaved: forecasts.json, forecast_metrics.json")

Loading daily data...


09:13:20 - cmdstanpy - INFO - Chain [1] start processing
09:13:20 - cmdstanpy - INFO - Chain [1] done processing


Events eligible for forecasting (>=30 active days): 201


09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:21 - cmdstanpy - INFO - Chain [1] done processing
09:13:21 - cmdstanpy - INFO - Chain [1] start processing
09:13:22 - cmdstanpy - INFO - Chain [1] done processing
09:13:22 - cmdstanpy - INFO - Chain [1] start processing
09:13:22 - cmdstanpy - INFO - Chain [1]

  Processed 10/201 events...


...

  Processed 180/201 events...


09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:49 - cmdstanpy - INFO - Chain [1] start processing
09:13:49 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1]

  Processed 190/201 events...


09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing
09:13:50 - cmdstanpy - INFO - Chain [1] start processing
09:13:50 - cmdstanpy - INFO - Chain [1] done processing


  Processed 200/201 events...

Forecasted 201 events successfully

Aggregate metrics:
  MAE:       0.005795
  RMSE:      0.009765
  Precision: 0.187
  Recall:    0.609
  F1:        0.243

Top 10 best predicted events (by F1):
     event_cluster  train_days  test_days       mae      rmse        f1
186            735         437        110  0.004328  0.006641  0.823529
160            659         724        181  0.023904  0.037330  0.805281
33             238        1267        317  0.020540  0.025870  0.710638
140            618         556        140  0.004903  0.007834  0.704762
132            593         526        132  0.006086  0.009909  0.662420
1                8         129         33  0.002802  0.003464  0.653061
168            684        1280        321  0.013995  0.021324  0.636559
161            663         296         75  0.006990  0.008247  0.615385
46             309         168         43  0.003922  0.005171  0.600000
179            720         135         34  0.000626  0

In [16]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime


def load_data(
    forecasts_path: str,
    metrics_path: str,
    daily_path: str,
    features_path: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    forecasts = pd.read_json(forecasts_path, lines=True)
    forecasts["ds"] = pd.to_datetime(forecasts["ds"])

    metrics = pd.read_json(metrics_path, lines=True)
    daily = pd.read_json(daily_path, lines=True)
    daily["date"] = pd.to_datetime(daily["date"])

    features = pd.read_json(features_path, lines=True)

    return forecasts, metrics, daily, features


def plot_event_forecast(
    forecasts: pd.DataFrame,
    daily: pd.DataFrame,
    features: pd.DataFrame,
    cluster_id: int,
) -> go.Figure:
    fc = forecasts[forecasts["event_cluster"] == cluster_id].sort_values("ds")
    feat = features[features["event_cluster"] == cluster_id].iloc[0]

    group = daily[daily["event_cluster"] == cluster_id].sort_values("date")
    full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    full_ts = group.set_index("date")["relative_intensity"].reindex(full_range, fill_value=0)

    split_date = fc["ds"].min()
    train_ts = full_ts[full_ts.index < split_date]

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=train_ts.index, y=train_ts.values,
        name="Train (факт)",
        line=dict(color="#4C9BE8", width=1.5),
        opacity=0.7,
    ))

    fig.add_trace(go.Scatter(
        x=fc["ds"], y=fc["actual"],
        name="Test (факт)",
        line=dict(color="#2ECC71", width=2),
    ))

    fig.add_trace(go.Scatter(
        x=fc["ds"], y=np.clip(fc["yhat"], 0, None),
        name="Прогноз",
        line=dict(color="#E74C3C", width=2, dash="dash"),
    ))

    fig.add_trace(go.Scatter(
        x=pd.concat([fc["ds"], fc["ds"].iloc[::-1]]),
        y=pd.concat([
            np.clip(fc["yhat_upper"], 0, None),
            np.clip(fc["yhat_lower"], 0, None).iloc[::-1],
        ]),
        fill="toself",
        fillcolor="rgba(231,76,60,0.1)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Доверительный интервал",
    ))

    fig.add_shape(type="line", x0=str(split_date), x1=str(split_date), y0=0, y1=1, yref="paper",
        line=dict(dash="dot", color="gray", width=1.5),
    )
    fig.add_annotation(x=str(split_date), y=1, yref="paper", text="Train / Test",
        showarrow=False, xanchor="left", font=dict(color="gray"),
    )


    pattern_ru = {
        "multi_burst": "многократные вспышки",
        "two_burst": "две вспышки",
        "long_tail": "длинный хвост",
        "slow_build": "медленный рост",
        "single_peak": "одиночный пик",
    }

    fig.update_layout(
        title=f"Событие #{cluster_id} — {pattern_ru.get(feat['pattern'], feat['pattern'])}<br>"
              f"<sup>Активных дней: {feat['active_days']} | Вспышек: {feat['n_bursts']} | "
              f"Пик: {str(feat['peak_date'])[:10]}</sup>",
        xaxis_title="Дата",
        yaxis_title="Относительная интенсивность",
        legend=dict(orientation="h", y=-0.2),
        height=450,
        template="plotly_white",
        hovermode="x unified",
    )

    return fig


def plot_metrics_distribution(metrics: pd.DataFrame) -> go.Figure:
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["MAE", "RMSE", "F1 Score"],
    )

    for col, name, color in [
        ("mae", "MAE", "#4C9BE8"),
        ("rmse", "RMSE", "#E74C3C"),
        ("f1", "F1", "#2ECC71"),
    ]:
        row_idx = ["mae", "rmse", "f1"].index(col) + 1
        fig.add_trace(
            go.Histogram(x=metrics[col], name=name, marker_color=color, opacity=0.8, nbinsx=30),
            row=1, col=row_idx,
        )

    fig.update_layout(
        title="Распределение метрик качества по всем событиям",
        height=350,
        template="plotly_white",
        showlegend=False,
    )
    return fig


def plot_pattern_distribution(features: pd.DataFrame) -> go.Figure:
    pattern_ru = {
        "multi_burst": "Многократные вспышки",
        "two_burst": "Две вспышки",
        "long_tail": "Длинный хвост",
        "slow_build": "Медленный рост",
        "single_peak": "Одиночный пик",
    }
    counts = features["pattern"].value_counts().reset_index()
    counts.columns = ["pattern", "count"]
    counts["pattern_ru"] = counts["pattern"].map(pattern_ru)

    fig = px.bar(
        counts,
        x="pattern_ru", y="count",
        color="pattern_ru",
        text="count",
        title="Распределение типов событий",
        labels={"pattern_ru": "Тип", "count": "Количество событий"},
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(showlegend=False, height=400)
    return fig


def plot_top_events_timeline(daily: pd.DataFrame, features: pd.DataFrame, top_n: int = 8) -> go.Figure:
    top_clusters = features.nlargest(top_n, "total_posts")["event_cluster"].tolist()

    fig = go.Figure()

    colors = px.colors.qualitative.Set1

    for i, cluster_id in enumerate(top_clusters):
        group = daily[daily["event_cluster"] == cluster_id].sort_values("date")
        full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
        ts = group.set_index("date")["relative_intensity"].reindex(full_range, fill_value=0)

        fig.add_trace(go.Scatter(
            x=ts.index,
            y=ts.values,
            name=f"Событие #{cluster_id}",
            line=dict(color=colors[i % len(colors)], width=1.5),
            opacity=0.8,
        ))

    fig.update_layout(
        title=f"Динамика топ-{top_n} событий по числу постов",
        xaxis_title="Дата",
        yaxis_title="Относительная интенсивность",
        height=500,
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", y=-0.25),
    )
    return fig


def plot_f1_vs_duration(metrics: pd.DataFrame, features: pd.DataFrame) -> go.Figure:
    merged = metrics.merge(features[["event_cluster", "total_days", "n_bursts", "pattern"]], on="event_cluster")

    pattern_ru = {
        "multi_burst": "Многократные вспышки",
        "two_burst": "Две вспышки",
        "long_tail": "Длинный хвост",
        "slow_build": "Медленный рост",
        "single_peak": "Одиночный пик",
    }
    merged["pattern_ru"] = merged["pattern"].map(pattern_ru)

    fig = px.scatter(
        merged,
        x="total_days",
        y="f1",
        color="pattern_ru",
        size="n_bursts",
        hover_data=["event_cluster", "mae", "rmse"],
        title="F1 vs продолжительность события",
        labels={"total_days": "Продолжительность (дней)", "f1": "F1 Score", "pattern_ru": "Тип"},
        template="plotly_white",
        color_discrete_sequence=px.colors.qualitative.Set2,
        height=450,
    )
    return fig


def run_visualization(
    forecasts_path: str = "forecasts.json",
    metrics_path: str = "forecast_metrics.json",
    daily_path: str = "events_daily.json",
    features_path: str = "events_features.json",
    output_html: str = f"reports/report-{datetime.now()}.html",
) -> None:
    print("Loading data...")
    forecasts, metrics, daily, features = load_data(
        forecasts_path, metrics_path, daily_path, features_path
    )

    best_clusters = metrics.nlargest(5, "f1")["event_cluster"].tolist()

    print("Building figures...")
    figs = []

    for cluster_id in best_clusters:
        fig = plot_event_forecast(forecasts, daily, features, cluster_id)
        figs.append(fig)

    figs.append(plot_top_events_timeline(daily, features))
    figs.append(plot_metrics_distribution(metrics))
    figs.append(plot_pattern_distribution(features))
    figs.append(plot_f1_vs_duration(metrics, features))

    print(f"Saving to {output_html}...")
    with open(output_html, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'>"
                "<title>Анализ информационных событий</title></head><body>")
        f.write("<h1 style='font-family:sans-serif;padding:20px'>Моделирование информационных событий</h1>")

        labels = (
            [f"Прогноз события #{c} (топ по F1)" for c in best_clusters]
            + ["Динамика топ-8 событий", "Метрики качества", "Типы событий", "F1 vs продолжительность"]
        )

        for label, fig in zip(labels, figs):
            f.write(f"<h2 style='font-family:sans-serif;padding:20px 20px 0'>{label}</h2>")
            f.write(fig.to_html(full_html=False, include_plotlyjs="cdn" if label == labels[0] else False))

        f.write("</body></html>")

    print(f"Done. Open {output_html} in browser.")


if __name__ == "__main__":
    run_visualization()

Loading data...
Building figures...
Saving to reports/report-2026-04-30 09:13:51.149902.html...
Done. Open reports/report-2026-04-30 09:13:51.149902.html in browser.
